In [1]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
import pandas as pd
from Propeller import Propeller
from JobParameters import AerodynamicParameters, AcousticParameters

with open("../10x7E.pkl", "rb") as f:
    blade_dict = pickle.load(f)

aerodynamic_params = AerodynamicParameters(
    prop_radius=blade_dict['tip_radius'],
    hub_radius=blade_dict['hub_radius'],
    n_blades=blade_dict['n_blades'],
    rpm=7000,
    v_inf=np.zeros(len(blade_dict['r'])),
    a_inf=343,
    rho=1.225,
    mu=1.81e-5,
)

acoustic_params = AcousticParameters(
    aero_params=aerodynamic_params,
    p_ref=2e-5,
    revolutions=5,
    num_obs_times_per_rev=100
)

propeller = Propeller(
    propeller_geometry=blade_dict,
    aero_params=aerodynamic_params,
    acoustic_params=acoustic_params
)

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

def generate_3d_von_karman(Lx, Ly, Lz, nx, ny, nz, sigma, L_scale):
    # 1. Setup wavenumbers
    kx = 2 * np.pi * np.fft.fftfreq(nx, d=Lx/nx)
    ky = 2 * np.pi * np.fft.fftfreq(ny, d=Ly/ny)
    kz = 2 * np.pi * np.fft.fftfreq(nz, d=Lz/nz)
    KX, KY, KZ = np.meshgrid(kx, ky, kz, indexing='ij')
    K = np.sqrt(KX**2 + KY**2 + KZ**2)
    
    # 2. Von Karman Energy Spectrum E(k)
    with np.errstate(divide='ignore', invalid='ignore'):
        # Standard isotropic formulation
        E_k = (1.4528 * sigma**2 * L_scale * (L_scale * K)**4) / (1 + (L_scale * K)**2)**(17/6)
        psd = E_k / (4 * np.pi * K**2)
    psd[0, 0, 0] = 0
    
    # 3. Generate the 3D box via IFFT
    white_noise = np.random.normal(size=(nx, ny, nz)) + 1j * np.random.normal(size=(nx, ny, nz))
    box = np.fft.ifftn(np.sqrt(psd) * white_noise).real
    return (box - np.mean(box)) / np.std(box) * sigma

# Propeller Diameter is 0.25m
D = 0.25

# 1. Grid Dimensions (Tighten the box around the prop)
Ly, Lz = 0.4, 0.4     # 40cm x 40cm cross section
Lx = 5.0              # 5 meters of "wind" to fly through

# 2. Resolution (Aim for ~1cm resolution)
ny, nz = 64, 64       # Gives ~0.6cm resolution (0.4 / 64)
nx = 512              # Higher resolution in X for smooth derivatives (dL/dt)

# 3. Physics
# sigma: 0.8 m/s is a ~10% turbulence intensity for a 8m/s inflow (standard)
sigma = 0.8           
# L_scale: Set to 1/2 of diameter to see spatial variation across blades
L_scale = 0.125       

u_box = generate_3d_von_karman(Lx, Ly, Lz, nx, ny, nz, sigma, L_scale)*0

In [3]:
# 2. Run the simulation
df = propeller.run_unsteady_simulation(
    u_box=u_box, 
    Lx=5.0, Ly=0.4, Lz=0.4, 
    dt=1e-6, 
    duration=0.5, 
    V_mean=0
)

# 3. Plot the unsteady response
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df['time'], df['thrust'], label='Unsteady Thrust')
ax.axhline(y=df['thrust'].mean(), color='r', linestyle='--', label='Mean Thrust')
ax.set_xlabel("Time (s)")
ax.set_ylabel("Thrust (N)")
ax.set_title("Propeller Thrust in Von Karman Turbulence (Pitt-Peters Dynamic Inflow)")
ax.legend()
plt.show()

Time: 0.0000s | Thrust: 7.073N | Torque: 0.1542Nm
Time: 0.0000s | Thrust: nanN | Torque: nanNm
Time: 0.0000s | Thrust: nanN | Torque: nanNm


KeyboardInterrupt: 